# 1. Dense Layers

The hidden layers of MLPs are called **dense** or **fully-connected layers** where the output of each neuron in the previous layer is taken as the inputs for a given neuron in the dense layer. In this notebook, we will mathematically construct dense layers and implement them using NumPy.

In [1]:
# Importing,
import numpy as np
from DeepLearn.activations import ReLU, SoftMax

### 1.1 A Single Neuron

Recall that we define a perceptron as,

$$
a = \phi (\mathbf{x}^T \mathbf{w} + b)
$$

where $d$ is the number of inputs, $\mathbf{x} = (x_1, x_2, ..., x_d)$ is the feature column vector, $\mathbf{w} = (w_1, w_2, ..., w_d)$ is the weight column vector, $b$ is the bias of the neuron and $a$ is its output. Given that $\mathbf{x}^T \mathbf{w}$ is simply the dot-product between $\mathbf{x}$ and $\mathbf{w}$, we can easily implement this,

In [2]:
# For a single neuron,
layer_input = np.array([1, 2, 3])
weights = np.array([0.2, 0.8, -0.5])
bias = 2

# Computing the output,
neuron_output = ReLU(np.dot(layer_input, weights) + bias)
neuron_output

2.3

### 1.2 Dense Layers

Now we begin constructing our _DenseLayer_ object to represent a fully connected layer where each neuron in the layer is connected to every neuron in the layer before it. Before we start defining our inputs, weights and biases, we must first decide on which convention to use. In pure mathematics, vectors are represented as column vectors. This is in contrast to some machine learning texts which represent vectors as row vectors. For our purposes, we will switch to the ML convention and use row-major order (treat vectors as row vectors by default) as it aligns more closely with our implementation goals. Let us say that we have $m$ neurons in the layer and $d$ inputs to each neuron (which corresponds to $d$ number of neurons in the previous layer). We define the input vector $\mathbf{x}$ of the layer as a row vector of length $d$ such that $\mathbf{x} = (x_1, x_2, ..., x_d)$. The next step is to define the weight matrix $\mathbf{W}$,

$$
\mathbf{W} = \begin{bmatrix} 
| & | & & | \\
\mathbf{w}_1 & \mathbf{w}_2 & \dots & \mathbf{w}_m \\
| & | & & |
\end{bmatrix}

=

\begin{bmatrix}
    w_{11} & w_{12} & w_{13} & \dots  & w_{1m} \\
    w_{21} & w_{22} & w_{23} & \dots  & w_{2m} \\
    \vdots & \vdots & \vdots & \ddots & \vdots \\
    w_{d1} & w_{d2} & w_{d3} & \dots  & w_{dm}
\end{bmatrix}
$$

Notice that we have arranged the weights of each neuron in the layer as a column vector $\mathbf{w}_i$ to construct the weight matrix of the layer. The bias of each neuron in the layer is contained within another column vector $\mathbf{b} = (b_1, b_2, ..., b_m)$. With these definitions, the foward-pass of the layer results in the row vector $\mathbf{z}$ that is given via,

$$
\mathbf{z} = \mathbf{x} \mathbf{W} + \mathbf{b}
$$

The activation function is then performed element-wise of each component of $\mathbf{z}$,

$$
\mathbf{a} = \phi(\mathbf{x} \mathbf{W} + \mathbf{b})
$$

Implementing this using NumPy,

In [3]:
# We have d=4 inputs fed into the layer,
layer_input = np.array([1, 2, 3, 2.5])

# Weights matrix for m=3 neurons,
weights = np.array([[0.2, 0.5, -0.26],
                    [0.8, -0.91, -0.27],
                    [-0.5, 0.26, 0.17],
                    [1.0, -0.5, 0.87]])

# Bias row vector for the m=3 neurons,
biases = np.array([2, 3, 0.5])

# Computing the output,
layer_output = ReLU(np.matmul(layer_input, weights) + biases)
layer_output

array([4.8  , 1.21 , 2.385])

Our foward-pass $\mathbf{z} = \mathbf{x} \mathbf{W} + \mathbf{b}$ is also the computation used by TensorFlow's `tf.keras.layers.Dense` object for fully connected layers: https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense. However, the equivalent in PyTorch (`torch.nn.linear`) uses $\mathbf{z} = \mathbf{x} \mathbf{W}^T + \mathbf{b}$ for its forward pass implementation. In PyTorch the weights of each neuron in the layer are arranged as a row vector $\mathbf{w}_i$ to construct the weight matrix $\mathbf{W}$ of the layer,

$$
\mathbf{W} = \begin{bmatrix} 
— & \mathbf{w_1} & — \\
— & \mathbf{w_2} & — \\
 & \vdots &  \\
— & \mathbf{w}_m & —
\end{bmatrix}
$$

For this reason, a transpose is used to ensure mathematical consistency with the TensorFlow implementation. 


### 1.3 Generalising to Batches

The ML convention makes it almost trivial to generalise our forward-pass to mutliple samples at a time (batches). We simply define our input matrix $\mathbf{X}$ as, 

$$
\mathbf{X} = \begin{bmatrix} 
— & \mathbf{x_1} & — \\
— & \mathbf{x_2} & — \\
 & \vdots &  \\
— & \mathbf{x}_B & —
\end{bmatrix}
$$

where we have treated each sample $\mathbf{x_i}$ as a row vector. The computation of the forward-pass remains the same with the exception of the biases being contained in a matrix $\mathbf{B}$ of the same dimensionality $()

$$
\mathbf{A} = \phi(\mathbf{X} \mathbf{W} + \mathbf{B})
$$

In a layer,

$$
\mathbf{A}^{(l)} = \phi(\mathbf{A}^{(l-1)} \mathbf{W}^{(l)} + \mathbf{B}^{(l)})
$$

In [4]:
input_array = np.array([[1, 2, 3, 2.5],
                        [2.5, 3, 2, 1],
                        [3, 2, 1, 2.5]])

weights = np.array([[0.2, 0.5, -0.26],
                    [0.8, -0.91, -0.27],
                    [-0.5, 0.26, 0.17],
                    [1.0, -0.5, 0.87]])

biases = [2, 3, 0.5]

# Computing the output,
output_array = ReLU(np.matmul(input_array, weights) + biases)
print(output_array)

[[4.8   1.21  2.385]
 [4.9   1.54  0.25 ]
 [6.2   1.69  1.525]]


### 1.4. DenseLayer Object

Let us now create the _DenseLayer_ object for our neural networks.

In [5]:
import numpy as np

class DenseLayer():
    """The DenseLayer class implements a dense or fully connected layer in which all connections from the previous layer are 
    fed as inputs to each neuron in the layer."""

    def __init__(self, n_inputs, n_neurons):
        """Constuctor method. The weights are initialised with random values between 0 and 1 while we construct
        the biases array as a zero array."""

        # Layer properties,
        self.n_inputs = n_inputs
        self.n_neurons = n_neurons
        self.output = None

        # Initialising layer parameters,
        self.weights = np.random.rand(n_inputs, n_neurons)
        self.biases = np.zeros(n_neurons)
        self.output = None

    def forward(self, input_array):
        """This method defines the forward propagation of the layer."""
        forward_pass = np.matmul(input_array, self.weights) + self.biases
        return forward_pass

    def __repr__(self):
        """Useful information for when the user prints the LayDense object."""
        repr_string = f"""
        Type: DENSE Layer,
        Number of Neurons: {self.n_neurons},
        Expected Input Array: (BATCH_SIZE, {self.n_inputs}),
        Output Array Shape: (BATCH_SIZE, {self.n_neurons}).
        """
        return repr_string

Now let us construct a very simple MLP,

In [6]:
# Creating the layers,
layer_1 = DenseLayer(2, 8)
layer_2 = DenseLayer(8, 2)

# Forward propagation,
x = np.array([[54, 1.52],
              [78, 1.78],
              [95, 1.83]])

x = layer_1.forward(x)
x = layer_2.forward(x)

In [8]:
softmax = SoftMax()
x = softmax.forward(x)